## Paso 1: Crear la tabla Silver

In [ ]:
from pyspark.sql import functions as F

src = spark.table("workspace.default.ventas_taco_bueno_clean")

def num(c):
    return F.expr(f"try_cast({c} AS DOUBLE)")

formatos = ["M/d/yyyy", "M/d/yyyy h:mm:ss a", "yyyy-MM-dd", "yyyy-MM-dd HH:mm:ss", "M/d/yy", "yyyyMMdd"]
fecha = F.coalesce(*[F.expr(f"try_to_timestamp(trim(Date), '{f}')") for f in formatos]).cast("date")

silver = (
    src.select(
        F.trim("Store_num").alias("store_id"),
        fecha.alias("business_date"),
        F.regexp_replace(F.trim("Check_Number"), r"\.0$", "").alias("check_number"),
        F.trim("Menu_Item").alias("menu_item"),
        F.trim("Report_Group").alias("report_group"),
        F.trim("Pop").alias("sales_channel"),
        F.trim("Transaction_Type").alias("transaction_type"),
        num("Quantity").alias("quantity"),
        num("Menu_Price").alias("menu_price"),
        F.abs(F.coalesce(num("Item_Discount"), F.lit(0.0))).alias("discount_amount"),
        num("Total").alias("check_total_reported"),
        F.hour(F.expr("try_to_timestamp(upper(trim(Begin_Time)), 'h:mm a')")).alias("order_hour"),
    )
    .filter(F.col("quantity") > 0)
    .filter(F.col("menu_price").isNotNull())
    .filter(F.col("check_number").isNotNull())
    .withColumn("order_id", F.concat_ws("|", "store_id", "business_date", "check_number"))
    .withColumn("gross_amount", F.col("quantity") * F.col("menu_price"))
    .withColumn("line_revenue", F.col("gross_amount") - F.col("discount_amount"))
    .withColumn("has_discount", F.col("discount_amount") > 0)
    .withColumn("day_of_week", F.date_format("business_date", "EEEE"))
)

silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.ventas_taco_bueno_silver")

print(f"Silver creada: {spark.table('workspace.default.ventas_taco_bueno_silver').count():,} filas")

## Paso 2: Data quality checks

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.default.ventas_taco_bueno_clean)   AS filas_tabla_original,
  COUNT(*)                                                          AS filas_silver,
  COUNT(DISTINCT order_id)                                          AS ordenes_reales,
  COUNT(DISTINCT check_number)                                      AS check_numbers_distintos,
  SUM(CASE WHEN business_date IS NULL THEN 1 ELSE 0 END)            AS fecha_no_reconocida,
  SUM(CASE WHEN order_hour IS NULL THEN 1 ELSE 0 END)               AS hora_no_reconocida,
  SUM(CASE WHEN store_id IS NULL THEN 1 ELSE 0 END)                 AS sin_tienda,
  SUM(CASE WHEN sales_channel IS NULL THEN 1 ELSE 0 END)            AS sin_canal,
  SUM(CASE WHEN discount_amount > gross_amount + 0.01 THEN 1 ELSE 0 END) AS descuento_mayor_al_precio
FROM workspace.default.ventas_taco_bueno_silver;

### Verificación

In [ ]:
%sql
WITH o AS (
  SELECT order_id,
         ROUND(SUM(line_revenue), 2)        AS suma_lineas,
         ROUND(MAX(check_total_reported), 2) AS total_ticket
  FROM workspace.default.ventas_taco_bueno_silver
  GROUP BY order_id
)
SELECT
  COUNT(*)                                                             AS ordenes,
  ROUND(100 * AVG(CASE WHEN ABS(suma_lineas - total_ticket) <= 0.05 THEN 1 ELSE 0 END), 2) AS pct_ordenes_que_cuadran,
  ROUND(SUM(total_ticket), 2)                                          AS revenue_correcto_por_ticket,
  ROUND(SUM(suma_lineas), 2)                                           AS revenue_por_lineas,
  (SELECT ROUND(SUM(check_total_reported), 2) FROM workspace.default.ventas_taco_bueno_silver) AS revenue_fase1_inflado
FROM o;

## Analysis A: Top Revenue & Best-Selling Menu Items

In [ ]:
%sql
SELECT
  menu_item,
  SUM(quantity)                  AS total_units,
  ROUND(SUM(gross_amount), 2)    AS gross_revenue_usd,
  ROUND(SUM(discount_amount), 2) AS discount_usd,
  ROUND(SUM(line_revenue), 2)    AS net_revenue_usd
FROM workspace.default.ventas_taco_bueno_silver
GROUP BY menu_item
ORDER BY net_revenue_usd DESC
LIMIT 10;

## Analysis B: Promotion Effectiveness (Discounted vs. Regular Price)

In [ ]:
%sql
SELECT
  CASE WHEN has_discount THEN 'Discounted' ELSE 'Regular Price' END AS pricing_type,
  COUNT(DISTINCT order_id)        AS total_orders,
  SUM(quantity)                   AS total_units,
  ROUND(AVG(quantity), 2)         AS avg_units_per_line,
  ROUND(SUM(discount_amount), 2)  AS total_discounts_usd,
  ROUND(SUM(line_revenue), 2)     AS net_revenue_usd
FROM workspace.default.ventas_taco_bueno_silver
GROUP BY has_discount;

## Analysis C: Sales Channel Performance

In [ ]:
%sql
SELECT
  sales_channel,
  COUNT(DISTINCT order_id)                       AS total_orders,
  SUM(quantity)                                  AS units_sold,
  ROUND(SUM(line_revenue), 2)                    AS total_sales_usd,
  ROUND(SUM(line_revenue) / COUNT(DISTINCT order_id), 2) AS avg_ticket_usd,
  ROUND(100 * COUNT(DISTINCT CASE WHEN has_discount THEN order_id END)
        / COUNT(DISTINCT order_id), 2)           AS pct_orders_with_promo
FROM workspace.default.ventas_taco_bueno_silver
GROUP BY sales_channel
ORDER BY total_sales_usd DESC;

## Analysis D: The "Lift" Test (Do Promos Increase Ticket Size?)

In [ ]:
%sql
WITH orders AS (
  SELECT order_id,
         SUM(line_revenue)  AS basket_usd,
         SUM(quantity)      AS units,
         MAX(CAST(has_discount AS INT)) = 1 AS has_promo
  FROM workspace.default.ventas_taco_bueno_silver
  GROUP BY order_id
)
SELECT
  CASE WHEN has_promo THEN 'Checks WITH Promos' ELSE 'Checks WITHOUT Promos' END AS ticket_type,
  COUNT(*)                                          AS total_orders,
  ROUND(SUM(basket_usd), 2)                         AS total_revenue_usd,
  ROUND(AVG(basket_usd), 2)                         AS avg_basket_usd,
  ROUND(PERCENTILE_APPROX(basket_usd, 0.5), 2)      AS median_basket_usd,
  ROUND(AVG(units), 2)                              AS avg_units
FROM orders
GROUP BY has_promo;

## Analysis E: Cannibalization (Discount Dependency)

In [ ]:
%sql
SELECT
  menu_item,
  SUM(quantity)                                              AS total_units,
  SUM(CASE WHEN has_discount THEN quantity ELSE 0 END)       AS units_on_discount,
  SUM(CASE WHEN NOT has_discount THEN quantity ELSE 0 END)   AS units_regular_price,
  ROUND(100 * SUM(CASE WHEN has_discount THEN quantity ELSE 0 END) / SUM(quantity), 2)
                                                             AS discount_dependency_pct
FROM workspace.default.ventas_taco_bueno_silver
GROUP BY menu_item
HAVING total_units > 1000
ORDER BY discount_dependency_pct DESC
LIMIT 15;

## Analysis F: Seasonality & Peak Hours

In [ ]:
%sql
SELECT
  order_hour,
  CASE WHEN order_hour = 0 THEN '12 AM' WHEN order_hour < 12 THEN CONCAT(order_hour, ' AM')
       WHEN order_hour = 12 THEN '12 PM' ELSE CONCAT(order_hour - 12, ' PM') END AS hour_label,
  COUNT(DISTINCT order_id)                 AS total_orders,
  ROUND(SUM(line_revenue), 2)              AS total_sales_usd
FROM workspace.default.ventas_taco_bueno_silver
WHERE order_hour IS NOT NULL
GROUP BY order_hour
ORDER BY order_hour;

### Día de la semana

In [ ]:
%sql
SELECT
  DAYOFWEEK(business_date)       AS day_num,
  day_of_week,
  COUNT(DISTINCT order_id)       AS total_orders,
  ROUND(SUM(line_revenue), 2)    AS total_sales_usd
FROM workspace.default.ventas_taco_bueno_silver
WHERE business_date IS NOT NULL
GROUP BY DAYOFWEEK(business_date), day_of_week
ORDER BY day_num;

## Analysis G: Halo Effect / Attach Rate

In [ ]:
%sql
WITH top_promo_items AS (
  SELECT menu_item AS promo_item
  FROM workspace.default.ventas_taco_bueno_silver
  WHERE has_discount
  GROUP BY menu_item
  ORDER BY COUNT(DISTINCT order_id) DESC
  LIMIT 5
),
promo_orders AS (
  SELECT DISTINCT s.order_id, s.menu_item AS promo_item
  FROM workspace.default.ventas_taco_bueno_silver s
  JOIN top_promo_items t ON s.menu_item = t.promo_item
  WHERE s.has_discount
),
companions AS (
  SELECT p.promo_item, s.menu_item AS companion_item,
         COUNT(DISTINCT s.order_id) AS orders_together
  FROM promo_orders p
  JOIN workspace.default.ventas_taco_bueno_silver s ON s.order_id = p.order_id
  WHERE s.menu_item <> p.promo_item
  GROUP BY p.promo_item, s.menu_item
)
SELECT * FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY promo_item ORDER BY orders_together DESC) AS rank_num
  FROM companions
)
WHERE rank_num <= 3
ORDER BY promo_item, rank_num;

## Analysis H: Promo ROI / Discount Efficiency

In [ ]:
%sql
SELECT
  menu_item,
  ROUND(SUM(line_revenue), 2)     AS revenue_usd,
  ROUND(SUM(discount_amount), 2)  AS discount_usd,
  ROUND(SUM(line_revenue) / NULLIF(SUM(discount_amount), 0), 2) AS revenue_per_discount_dollar,
  ROUND(100 * SUM(CASE WHEN has_discount THEN quantity ELSE 0 END) / SUM(quantity), 2) AS discount_dependency_pct
FROM workspace.default.ventas_taco_bueno_silver
GROUP BY menu_item
HAVING discount_usd > 0 AND SUM(quantity) > 1000
ORDER BY revenue_per_discount_dollar ASC
LIMIT 15;